# Correct a learned behavior (DAgger from a notebook)

Pause a policy that's currently running, take over by hand via the gello, record the correction, and save it for retraining. The cells use server-side ticket cancellation, so it doesn't matter whether the policy was started by this notebook, by your own EC2 process, or by anything else — `cancel_active_behaviours()` looks up what's running and stops it.

**Before you start:**

- Plug your gello into your machine and start the standalone gello-ros2 node in a separate terminal: `CONFIG=gello-<yourname>.yaml pixi run gello`. The notebook does **not** manage the gello hardware — it expects the joint_states topics to already be flowing on the network.
- Confirm DDS is healthy across the two machines (Cyclone DDS, subnet discovery — same setup we used to test cross-machine teleop with `launch_sdk`).
- Select the R2 SDK kernel before running.

**Typical use:** run the **Setup** cell once. Then run **Intervene** when you want to take over a running policy, drive the gello to correct it, and finally **Save** or **Discard** the episode.

In [ ]:
from r2_labs import client, rpc_api


entry_prefix = "dagger_rectify_open_latch"
align_timeout_seconds = 1.0
align_period_seconds = 0.0

host = "localhost"
robot = client.Robot(
    server_address=f"tcp://{host}:{rpc_api.DEFAULT_PORT}",
    query_server_address=f"tcp://{host}:{rpc_api.DEFAULT_QUERY_PORT}",
    training_server_address=f"tcp://{host}:{rpc_api.DEFAULT_MODEL_TRAINER_PORT}",
)

_ACTIVE_STATUSES = (rpc_api.TicketStatus.PENDING, rpc_api.TicketStatus.RUNNING)


def _build_behavior_query(model: str) -> rpc_api.ExecuteLearnedBehaviorQuery:
  return rpc_api.ExecuteLearnedBehaviorQuery(
      model_id=model,
      obs_history_len=1,
      buffer_actions=20,
      action_offset=2,
      action_key="action",
  )


def cancel_active_behaviours() -> int:
  """Cancel every PENDING/RUNNING behaviour ticket. Returns the count cancelled."""
  active = [
      t for t in robot.behaviour.list_tickets().tickets
      if t.status in _ACTIVE_STATUSES
  ]
  for t in active:
    print(f"  cancelling {t.behaviour_type} (ticket {t.ticket_id[:12]}...)")
    robot.behaviour.cancel_behaviour(t.ticket_id)
  return len(active)


def _align_leader() -> None:
  robot.behaviour.align_leader_with_follower(
      timeout_seconds=align_timeout_seconds,
      threshold=0.1,
      period_seconds=align_period_seconds,
  ).result()


print(f"Connected to {host}")


## 1. Start the policy (optional)

Skip if another process (e.g. your EC2 runner) is launching `execute_learned_behavior`. Use this cell if you want the notebook itself to drive the policy.

In [ ]:
model_id = "DCAM#tender-engineer-160"

robot.episode_observer.set_is_human(False)
robot.episode_observer.start()

robot.exec_mode.set_execution_mode(rpc_api.ExecutionMode.READY)
robot.behaviour.execute_learned_behavior(_build_behavior_query(model_id))
print(f"Policy running: {model_id}")


## 2. Intervene

The main cell. Cancels whatever behavior is currently running on the robot (whether started here or by another process), aligns your gello to the current follower pose, and switches to teleop with `is_human=True`. Recording continues from that point as a human-driven segment.

When you're done correcting, run cell **3a / 3b** to end the episode.

In [ ]:
print("Cancelling active behaviour(s)...")
if cancel_active_behaviours() == 0:
  print("  (none were running)")

print("Aligning your gello with the follower arm...")
_align_leader()

robot.exec_mode.set_execution_mode(rpc_api.ExecutionMode.DATA_COLLECTION_TELEOP)
robot.episode_observer.set_is_human(True)
robot.episode_observer.start()  # idempotent — works if observer is already running
print("Teleop active. Drive the gello to correct the behaviour.")


## 3. End the episode

Run **3a** to keep the data, **3b** to mark it discarded (saved but won't be trained on). Either way, any active behavior is cancelled and the robot returns to `READY`.

In [ ]:
# 3a — save
cancel_active_behaviours()
robot.episode_observer.stop()
robot.exec_mode.set_execution_mode(rpc_api.ExecutionMode.READY)

robot.episode_observer.set_task_description("DAgger correction")
robot.episode_observer.save(entry_prefix=entry_prefix)
print("Episode saved.")


In [ ]:
# 3b — discard
cancel_active_behaviours()
robot.episode_observer.stop()
robot.exec_mode.set_execution_mode(rpc_api.ExecutionMode.READY)

robot.episode_observer.set_task_description("DAgger DISCARDED")
robot.episode_observer.save(entry_prefix=f"discarded_{entry_prefix}")
print("Episode saved as discarded.")


## Emergency cleanup

Safe to run any time — if a cell hung, the kernel got into a weird state, or you just want to bail. Cancels every active behavior, drops any unsaved recording, and puts the robot back into `READY`.

In [ ]:
cancel_active_behaviours()

try:
  robot.episode_observer.stop()
  robot.episode_observer.discard()
except Exception as e:
  print(f"observer cleanup: {e}")

robot.exec_mode.set_execution_mode(rpc_api.ExecutionMode.READY)
print("Robot in READY. Episode (if any) discarded.")
